# Notebook 31 — Resumable IMERG event collection

This is Phase 2. Run Notebook 30 first to create the Drive event plan and authenticate with NASA Earthdata. This notebook collects only IMERG, event by event. It immediately reduces each downloaded field to the Shinoda JPCZ polygon and coastal-wedge area means, checkpoints the compact half-hourly rates, and then checkpoints one complete precipitation summary row per event.

The original global NASA HDF files are not duplicated to Drive. They are read, reduced, and removed from temporary Colab storage. The Drive rate cache, event table, and plan are the backed-up analysis dataset.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git'
BRANCH = 'codex/notebook16-pcolormesh'
REPO_DIR = '/content/JPCZcatalog'
FORCE_REFRESH_REPO = True
DRIVE_ROOT = Path('/content/drive/MyDrive/JPCZcatalog_outputs')
# Accept a raw Git URL even if it was accidentally pasted as a Markdown link.
if REPO_URL.startswith('[') and '](' in REPO_URL and REPO_URL.endswith(')'):
    REPO_URL = REPO_URL.rsplit('](', 1)[1][:-1]
if not REPO_URL.startswith('https://'):
    raise ValueError(f'REPO_URL must be a raw https Git URL, not {REPO_URL!r}')

from google.colab import drive
drive.mount('/content/drive')
if FORCE_REFRESH_REPO and Path(REPO_DIR).exists():
    shutil.rmtree(REPO_DIR)
if not Path(REPO_DIR).exists():
    clone = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, REPO_DIR], text=True, capture_output=True)
    if clone.returncode:
        raise RuntimeError(f'Git clone failed for {REPO_URL}:\n{clone.stderr}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO_DIR}/requirements-colab.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO_DIR], check=True)
os.chdir(REPO_DIR)
if f'{REPO_DIR}/src' not in sys.path:
    sys.path.insert(0, f'{REPO_DIR}/src')
print('Repository branch:', BRANCH)

In [ ]:
import gc
import numpy as np
import pandas as pd

from jpcz_catalog.config import BoundingBox, JPCZ_POLYGON_VERTICES
from jpcz_catalog.imerg import parse_imerg_granule_start, read_precipitation_cal_subset, region_mean_rates
from jpcz_catalog.imerg_workflow import atomic_csv, completed_event_ids, event_precipitation_metrics, merge_checkpoint, read_checkpoint, write_event_plan

# Keep this small initially. Rerun the collection cell to resume.
EVENTS_PER_COLLECTION_RUN = 1
MIN_TEMPORAL_COVERAGE = 0.90
DELETE_LOCAL_GRANULES_AFTER_REDUCTION = True
RATE_CHECKPOINT_EVERY_GRANULES = 12

COASTAL_WEDGE_VERTICES = ((133.05, 35.55), (136.05, 35.55), (139.55, 39.00), (139.55, 42.55))
REGIONS = {'jpcz_polygon': JPCZ_POLYGON_VERTICES, 'coastal_wedge': COASTAL_WEDGE_VERTICES}
IMERG_READ_DOMAIN = BoundingBox(lon_min=128.0, lon_max=141.0, lat_min=35.0, lat_max=43.0)

DRIVE_ANALYSIS_DIR = DRIVE_ROOT / 'imerg_precipitation_convergence'
PLAN_PATH = DRIVE_ANALYSIS_DIR / 'imerg_event_collection_plan.csv'
RATE_PATH = DRIVE_ANALYSIS_DIR / 'imerg_regional_halfhourly_rates.csv'
EVENT_PATH = DRIVE_ANALYSIS_DIR / 'imerg_event_regional_precipitation.csv'
ATTEMPT_LOG_PATH = DRIVE_ANALYSIS_DIR / 'imerg_collection_attempt_log.csv'
LOCAL_GRANULE_DIR = Path('/content/imerg_event_granules')

if not PLAN_PATH.exists():
    raise FileNotFoundError('Drive event plan is missing. Run Notebook 30 first.')
events = pd.read_csv(PLAN_PATH, parse_dates=['event_start', 'event_end', 'event_peak', 'precip_window_start', 'precip_window_end_exclusive'])
print(f'Loaded Drive collection plan: {len(events)} IMERG-eligible events.')
display(events.head())

## Authenticate

Run this once per new Colab session before collection.

In [ ]:
import earthaccess
earthaccess.login()
earthdata_authenticated = True
print('Earthdata authentication completed. You may now run the collection cell.')

## Collect and back up the next event batch

This is the only cell that retrieves NASA data. It is resumable: completed events are skipped, partial regional rates are reused, and each event writes its result to Drive before the next event begins.

In [ ]:
if not globals().get('earthdata_authenticated', False):
    raise RuntimeError('Run the authentication cell first.')

event_metrics = read_checkpoint(EVENT_PATH, parse_dates=('event_peak',))
rates = read_checkpoint(RATE_PATH, parse_dates=('time',))
attempt_log = read_checkpoint(ATTEMPT_LOG_PATH, parse_dates=('attempt_utc',))
complete_ids = completed_event_ids(event_metrics)
pending = events.loc[~events['event_id'].isin(complete_ids)].copy()
batch = pending if EVENTS_PER_COLLECTION_RUN is None else pending.head(int(EVENTS_PER_COLLECTION_RUN))
print(f'Collection inventory: {len(complete_ids)} complete; {len(pending)} pending; processing {len(batch)} now.')

for batch_position, event in enumerate(batch.itertuples(index=False), start=1):
    attempt = {'event_id': event.event_id, 'attempt_utc': pd.Timestamp.utcnow(), 'status': 'started', 'detail': ''}
    try:
        expected = pd.date_range(event.precip_window_start, event.precip_window_end_exclusive, freq='30min', inclusive='left')
        existing_times = pd.DatetimeIndex(rates['time']) if 'time' in rates else pd.DatetimeIndex([])
        wanted = set(expected.difference(existing_times))
        print(f'Event {batch_position}/{len(batch)}: {event.event_id}; {len(wanted)}/{len(expected)} half-hours need retrieval.')
        fresh_rows = []
        if wanted:
            local_event_dir = LOCAL_GRANULE_DIR / event.event_id
            local_event_dir.mkdir(parents=True, exist_ok=True)
            results = earthaccess.search_data(
                short_name='GPM_3IMERGHH', version='07',
                temporal=(pd.Timestamp(event.precip_window_start).isoformat(), pd.Timestamp(event.precip_window_end_exclusive).isoformat()),
                count=500,
            )
            if not results:
                raise RuntimeError('NASA returned no IMERG granules for this event window.')
            for granule_number, granule in enumerate(results, start=1):
                downloaded_paths = earthaccess.download([granule], local_path=local_event_dir, threads=1)
                for file_path in downloaded_paths:
                    file_path = Path(file_path)
                    try:
                        timestamp = pd.Timestamp(parse_imerg_granule_start(file_path))
                        if timestamp in wanted:
                            rate_field = read_precipitation_cal_subset(file_path, domain=IMERG_READ_DOMAIN)
                            fresh_rows.append({'time': timestamp, **{f'{name}_rate_mm_hr': value for name, value in region_mean_rates(rate_field, REGIONS).items()}})
                    finally:
                        if DELETE_LOCAL_GRANULES_AFTER_REDUCTION:
                            file_path.unlink(missing_ok=True)
                if fresh_rows and len(fresh_rows) % RATE_CHECKPOINT_EVERY_GRANULES == 0:
                    rates = merge_checkpoint(rates, pd.DataFrame(fresh_rows), key='time')
                    atomic_csv(rates, RATE_PATH)
                    fresh_rows = []
                    print(f'  rate cache checkpointed after {granule_number} granules.')
            if fresh_rows:
                rates = merge_checkpoint(rates, pd.DataFrame(fresh_rows), key='time')
                atomic_csv(rates, RATE_PATH)
        metric = pd.DataFrame([event_precipitation_metrics(event, rates, region_names=tuple(REGIONS), minimum_coverage=MIN_TEMPORAL_COVERAGE)])
        if metric.loc[0, 'status'] != 'ok':
            raise RuntimeError('IMERG temporal coverage was below the required threshold; the event will remain pending.')
        event_metrics = merge_checkpoint(event_metrics, metric, key='event_id')
        atomic_csv(event_metrics, EVENT_PATH)
        attempt.update(status='complete', detail=f"{int(metric.loc[0, 'imerg_expected_halfhours'])} expected half-hours")
        print(f'  saved completed event to Drive: {event.event_id}')
    except Exception as error:
        attempt.update(status='failed', detail=f'{type(error).__name__}: {error}')
        print(f'  event was not marked complete: {attempt["detail"]}')
    finally:
        attempt_log = pd.concat([attempt_log, pd.DataFrame([attempt])], ignore_index=True)
        atomic_csv(attempt_log, ATTEMPT_LOG_PATH)
        gc.collect()

event_metrics = read_checkpoint(EVENT_PATH, parse_dates=('event_peak',))
plan = write_event_plan(events, event_metrics, path=PLAN_PATH)
print('Drive plan updated after this batch:')
display(plan['collection_status'].value_counts().rename_axis('status').reset_index(name='event_count'))
display(attempt_log.tail())